# Rebuild company-year features (staged via local SSD)

Stages the source parquet (clean BODACC + INPI + INSEE + financials) from Drive to the runtime's local SSD, runs the feature builder against the local copy (much faster I/O than Drive FUSE), then syncs the new features back to Drive — with optional backup of the previous version.

Expected total time: **~30–45 min** end to end on a High-RAM CPU runtime.
- Stage copy: ~10–15 min (limited by Drive read speed)
- Build (DuckDB compute): ~15–25 min
- Sync back to Drive: ~10–15 min
- **vs ~60–90 min** if you built directly on Drive.

Schema target: **`3.0-trajectory-sector`** (52 features, up from 33 — adds legal-event trajectory, filing-pattern, sector context).

In [ ]:
import os, sys, subprocess
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'data-extraction'
REPO_DIR = Path('/content/pfein')
BACKEND_DIR = REPO_DIR / 'back_end'

if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)])
subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', 'origin'])
subprocess.check_call(['git', '-C', str(REPO_DIR), 'switch', BRANCH])
subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH])
os.chdir(BACKEND_DIR)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'collabs/requirements-colab.txt'])

# Quick check that the new feature SQL is on the pulled branch.
script_text = (BACKEND_DIR / 'app' / 'tools' / 'build_company_year_features.py').read_text(encoding='utf-8')
print('HEAD :', subprocess.check_output(['git', '-C', str(REPO_DIR), 'log', '--oneline', '-1']).decode().strip())
print('Has trajectory features  :', 'legal_events_acceleration_6m' in script_text)
print('Has sector context features:', 'naf2_continuity_failure_rate' in script_text)
if 'legal_events_acceleration_6m' not in script_text:
    raise SystemExit('Stale clone — delete /content/pfein and re-run this cell.')

## 1. Configure

`MAX_COMPANIES = None` runs on the full population (~191M rows output). Set to e.g. `500_000` for a fast smoke test first (~10 min instead of ~40).

In [ ]:
DRIVE_DATA_LAKE = Path('/content/drive/MyDrive/PFE ML Data/pfe_data/data-lake')
LOCAL_DATA_LAKE = Path('/content/data-lake')

START_YEAR    = 2017
END_YEAR      = 2024
MAX_COMPANIES = None       # None = full rebuild; integer = smoke test on a subset
BACKUP_OLD_FEATURES = True # rename Drive features/ to features_backup_<ts>/ before overwriting

print('Drive data lake :', DRIVE_DATA_LAKE)
print('Local staging   :', LOCAL_DATA_LAKE)
print(f'Years {START_YEAR}-{END_YEAR} | max_companies = {MAX_COMPANIES} | backup_old = {BACKUP_OLD_FEATURES}')

## 2. Stage source data Drive → local SSD

Copies the `clean/` sub-trees the feature builder reads. If a `clean/` source is missing, the builder will silently fall back to its `raw/` counterpart (we'd want to avoid that — clean is canonical).

In [ ]:
import shutil, time

SOURCES = [
    'clean/company_identity',
    'clean/legal_events',
    'clean/formalities_events',
    'clean/annual_accounts',
    'clean/financials',
]

LOCAL_DATA_LAKE.mkdir(parents=True, exist_ok=True)

for sub in SOURCES:
    src = DRIVE_DATA_LAKE / sub
    dst = LOCAL_DATA_LAKE / sub
    if not src.exists():
        print(f'!! MISSING in Drive: {src} -- builder will fall back to raw/ if present')
        continue
    if dst.exists():
        print(f'   Removing existing staged copy: {dst}')
        shutil.rmtree(dst)
    print(f'-> {sub}', end=' ', flush=True)
    t0 = time.time()
    shutil.copytree(src, dst)
    elapsed = time.time() - t0
    size = subprocess.check_output(['du', '-sh', str(dst)]).decode().split()[0]
    print(f' ({size}, {elapsed:.0f}s)')

total = subprocess.check_output(['du', '-sh', str(LOCAL_DATA_LAKE)]).decode().split()[0]
print(f'\nTotal staged: {total}')

## 3. Build features against the local data lake

Output goes to `LOCAL_DATA_LAKE/features/` (also fast). We sync it back to Drive in the next step.

In [ ]:
import time, shlex

cmd = [
    sys.executable, '-u', '-m', 'app.tools.build_company_year_features',
    '--data-lake-dir', str(LOCAL_DATA_LAKE),
    '--start-year', str(START_YEAR),
    '--end-year', str(END_YEAR),
    '--overwrite',
]
if MAX_COMPANIES:
    cmd += ['--max-companies', str(MAX_COMPANIES)]

print(' '.join(shlex.quote(p) for p in cmd), '\n')
t0 = time.time()
subprocess.run(cmd, check=True, cwd=str(BACKEND_DIR))
print(f'\nFeature build finished in {(time.time() - t0) / 60:.1f} min')

# Show the produced manifest so we can confirm the new schema landed.
import json
manifest_path = LOCAL_DATA_LAKE / 'features' / 'company_year_features' / '_manifest.json'
if manifest_path.exists():
    print('\nManifest:', json.dumps(json.loads(manifest_path.read_text()), indent=2))

In [ ]:
## 4. Sync the new features back to Drive (with optional backup)
from datetime import datetime
import duckdb

local_features = LOCAL_DATA_LAKE / 'features'
drive_features = DRIVE_DATA_LAKE / 'features'

if not local_features.exists():
    raise SystemExit(f'No features produced at {local_features} -- did the build cell run?')

if drive_features.exists():
    if BACKUP_OLD_FEATURES:
        backup = drive_features.with_name(f"features_backup_{datetime.now().strftime('%Y%m%d-%H%M%S')}")
        print(f'Backing up old Drive features to: {backup.name}')
        shutil.move(str(drive_features), str(backup))
    else:
        print('Removing old Drive features (no backup) ...')
        shutil.rmtree(drive_features)

print(f'Syncing {local_features} -> {drive_features} ...', flush=True)
t0 = time.time()
shutil.copytree(local_features, drive_features)
print(f'Done in {(time.time() - t0) / 60:.1f} min')

# Validate: row count and column count of the new features parquet.
glob = (drive_features / 'company_year_features' / '**' / '*.parquet').as_posix().replace("'", "''")
con = duckdb.connect()
try:
    n_rows = con.execute(f"SELECT count(*) FROM read_parquet('{glob}', union_by_name=true)").fetchone()[0]
    n_cols = len(con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{glob}', union_by_name=true) LIMIT 0"
    ).fetchall())
finally:
    con.close()
print(f'\nNew features parquet: {n_rows:,} rows, {n_cols} columns.')
print('You can now run pfe_ml_v1_train.ipynb against the new feature set.')